<a href="https://colab.research.google.com/github/TTS-77/Cyclone-AI/blob/main/Notebook/Intensity_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("cyclone_train_2000_2018.csv")
val_df = pd.read_csv("cyclone_validation_2019_2021.csv")
test_df = pd.read_csv("cyclone_test_2022_2025.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

FileNotFoundError: [Errno 2] No such file or directory: 'cyclone_train_2000_2018.csv'

In [ ]:
#Intensity Model

In [ ]:
# ===== INTENSITY MODEL — WIND PREPARATION =====

# Features used by the model
FEATURES = [
    "LAT",
    "LON",
    "u10",
    "v10",
    "wind_speed",
    "t2m_C",
    "d2m_C",
    "temp_dewpoint_diff",
    "msl_hPa",
    "sst_C",
    "movement_speed_kmh"
]

TARGET = "target_wind"

# Keep only rows where the actual wind intensity is known
train_wind = train_df.dropna(subset=[TARGET]).copy()
val_wind = val_df.dropna(subset=[TARGET]).copy()
test_wind = test_df.dropna(subset=[TARGET]).copy()

# Create X and y
X_train = train_wind[FEATURES]
y_train = train_wind[TARGET]

X_val = val_wind[FEATURES]
y_val = val_wind[TARGET]

X_test = test_wind[FEATURES]
y_test = test_wind[TARGET]

print("Wind intensity data prepared.")
print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")
print("Training shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Test shape:", X_test_scaled.shape)

In [ ]:
# ===== FIX WIND TARGET DATA TYPE =====

# Convert target to numeric
# Invalid/blank values become NaN
train_wind[TARGET] = pd.to_numeric(
    train_wind[TARGET],
    errors="coerce"
)

val_wind[TARGET] = pd.to_numeric(
    val_wind[TARGET],
    errors="coerce"
)

test_wind[TARGET] = pd.to_numeric(
    test_wind[TARGET],
    errors="coerce"
)

# Remove any rows that became NaN
train_wind = train_wind.dropna(subset=[TARGET])
val_wind = val_wind.dropna(subset=[TARGET])
test_wind = test_wind.dropna(subset=[TARGET])

# Recreate X and y
X_train = train_wind[FEATURES]
y_train = train_wind[TARGET]

X_val = val_wind[FEATURES]
y_val = val_wind[TARGET]

X_test = test_wind[FEATURES]
y_test = test_wind[TARGET]

print("Wind target fixed.")
print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))

In [ ]:
# ===== FINAL PREPARATION BEFORE MODEL TRAINING =====

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Make sure all feature columns are numeric
for col in FEATURES:
    train_wind[col] = pd.to_numeric(train_wind[col], errors="coerce")
    val_wind[col] = pd.to_numeric(val_wind[col], errors="coerce")
    test_wind[col] = pd.to_numeric(test_wind[col], errors="coerce")

# Recreate X and y
X_train = train_wind[FEATURES]
y_train = train_wind[TARGET]

X_val = val_wind[FEATURES]
y_val = val_wind[TARGET]

X_test = test_wind[FEATURES]
y_test = test_wind[TARGET]

# Impute missing feature values using TRAINING data only
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

# Scale using TRAINING data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Final wind-intensity data prepared.")
print("Training:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

In [ ]:
# ===== RANDOM FOREST BASELINE =====

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Train the model
model.fit(X_train_scaled, y_train)

# Predict validation wind intensity
y_val_pred = model.predict(X_val_scaled)

# Calculate metrics
mae = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2 = r2_score(y_val, y_val_pred)

print("===== RANDOM FOREST BASELINE =====")
print(f"Validation MAE  : {mae:.2f} knots")
print(f"Validation RMSE : {rmse:.2f} knots")
print(f"Validation R²   : {r2:.4f}")

In [ ]:
# ===== FEATURE IMPORTANCE =====

importance_df = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("===== FEATURE IMPORTANCE =====")
print(importance_df.to_string(index=False))

In [ ]:
# ===== ABLATION TEST: REMOVE MSL =====

FEATURES_NO_MSL = [
    "LAT",
    "LON",
    "u10",
    "v10",
    "wind_speed",
    "t2m_C",
    "d2m_C",
    "temp_dewpoint_diff",
    "sst_C",
    "movement_speed_kmh"
]

# Create datasets without MSL
X_train_no_msl = train_wind[FEATURES_NO_MSL]
X_val_no_msl = val_wind[FEATURES_NO_MSL]

# Impute using training data only
imputer_no_msl = SimpleImputer(strategy="median")

X_train_no_msl = imputer_no_msl.fit_transform(X_train_no_msl)
X_val_no_msl = imputer_no_msl.transform(X_val_no_msl)

# Scale using training data only
scaler_no_msl = StandardScaler()

X_train_no_msl = scaler_no_msl.fit_transform(X_train_no_msl)
X_val_no_msl = scaler_no_msl.transform(X_val_no_msl)

# Train second Random Forest
model_no_msl = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model_no_msl.fit(X_train_no_msl, y_train)

# Validation prediction
y_val_pred_no_msl = model_no_msl.predict(X_val_no_msl)

# Metrics
mae_no_msl = mean_absolute_error(
    y_val,
    y_val_pred_no_msl
)

rmse_no_msl = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_pred_no_msl
    )
)

r2_no_msl = r2_score(
    y_val,
    y_val_pred_no_msl
)

print("===== WITHOUT MSL =====")
print(f"Validation MAE  : {mae_no_msl:.2f} knots")
print(f"Validation RMSE : {rmse_no_msl:.2f} knots")
print(f"Validation R²   : {r2_no_msl:.4f}")

In [ ]:
# ===== GRADIENT BOOSTING BASELINE =====

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# Train
gb_model.fit(X_train_scaled, y_train)

# Validation prediction
y_val_pred_gb = gb_model.predict(X_val_scaled)

# Evaluation
gb_mae = mean_absolute_error(y_val, y_val_pred_gb)
gb_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_gb))
gb_r2 = r2_score(y_val, y_val_pred_gb)

print("===== GRADIENT BOOSTING =====")
print(f"Validation MAE  : {gb_mae:.2f} knots")
print(f"Validation RMSE : {gb_rmse:.2f} knots")
print(f"Validation R²   : {gb_r2:.4f}")

print("\n===== MODEL COMPARISON =====")
print(f"Random Forest   → MAE: 13.07 | RMSE: 18.51 | R²: 0.6236")
print(f"Gradient Boost  → MAE: {gb_mae:.2f} | RMSE: {gb_rmse:.2f} | R²: {gb_r2:.4f}")

In [ ]:
# ===== RANDOM FOREST HYPERPARAMETER TUNING =====

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Different configurations to test
configs = [
    {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },
    {
        "n_estimators": 300,
        "max_depth": 20,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },
    {
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 2,
        "max_features": 1.0
    },
    {
        "n_estimators": 300,
        "max_depth": 20,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 500,
        "max_depth": 20,
        "min_samples_leaf": 2,
        "max_features": 1.0
    }
]

results = []

for i, config in enumerate(configs, start=1):

    rf = RandomForestRegressor(
        **config,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train_scaled, y_train)

    pred = rf.predict(X_val_scaled)

    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val, pred)

    results.append({
        "Model": i,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        **config
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "MAE"
).reset_index(drop=True)

print("===== RANDOM FOREST TUNING RESULTS =====")
print(results_df.to_string(index=False))

In [ ]:
# ===== SAFE EVOLUTION FEATURES =====

def add_safe_features(data):
    data = data.copy()

    data["ISO_TIME"] = pd.to_datetime(data["ISO_TIME"])

    data = data.sort_values(
        ["SID", "ISO_TIME"]
    ).reset_index(drop=True)

    # Cyclone age
    first_time = (
        data.groupby("SID")["ISO_TIME"]
        .transform("min")
    )

    data["cyclone_age_hours"] = (
        data["ISO_TIME"] - first_time
    ).dt.total_seconds() / 3600

    # Month
    data["month"] = data["ISO_TIME"].dt.month

    # Cyclic month encoding
    data["month_sin"] = np.sin(
        2 * np.pi * data["month"] / 12
    )

    data["month_cos"] = np.cos(
        2 * np.pi * data["month"] / 12
    )

    return data

In [ ]:
train_wind = add_safe_features(train_wind)
val_wind = add_safe_features(val_wind)
test_wind = add_safe_features(test_wind)

print("Safe evolution features added.")

In [ ]:
# ===== SAFE EVOLUTION FEATURES =====

def add_safe_features(data):
    data = data.copy()

    data["ISO_TIME"] = pd.to_datetime(data["ISO_TIME"])

    data = data.sort_values(
        ["SID", "ISO_TIME"]
    ).reset_index(drop=True)

    # Cyclone age
    first_time = (
        data.groupby("SID")["ISO_TIME"]
        .transform("min")
    )

    data["cyclone_age_hours"] = (
        data["ISO_TIME"] - first_time
    ).dt.total_seconds() / 3600

    # Month
    data["month"] = data["ISO_TIME"].dt.month

    # Cyclic month encoding
    data["month_sin"] = np.sin(
        2 * np.pi * data["month"] / 12
    )

    data["month_cos"] = np.cos(
        2 * np.pi * data["month"] / 12
    )

    return data

In [ ]:
train_wind = add_safe_features(train_wind)
val_wind = add_safe_features(val_wind)
test_wind = add_safe_features(test_wind)

print("Safe evolution features added.")

In [ ]:
# ===== 14-FEATURE INTENSITY DATA =====

FEATURES_14 = [
    "LAT",
    "LON",
    "u10",
    "v10",
    "wind_speed",
    "t2m_C",
    "d2m_C",
    "temp_dewpoint_diff",
    "msl_hPa",
    "sst_C",
    "movement_speed_kmh",
    "cyclone_age_hours",
    "month_sin",
    "month_cos"
]

TARGET = "target_wind"

# Create X and y
X_train_14 = train_wind[FEATURES_14]
y_train_14 = train_wind[TARGET]

X_val_14 = val_wind[FEATURES_14]
y_val_14 = val_wind[TARGET]

X_test_14 = test_wind[FEATURES_14]
y_test_14 = test_wind[TARGET]

# Make everything numeric
for col in FEATURES_14:
    X_train_14[col] = pd.to_numeric(X_train_14[col], errors="coerce")
    X_val_14[col] = pd.to_numeric(X_val_14[col], errors="coerce")
    X_test_14[col] = pd.to_numeric(X_test_14[col], errors="coerce")

# Impute using TRAINING data only
imputer_14 = SimpleImputer(strategy="median")

X_train_14_imp = imputer_14.fit_transform(X_train_14)
X_val_14_imp = imputer_14.transform(X_val_14)
X_test_14_imp = imputer_14.transform(X_test_14)

# Scale using TRAINING data only
scaler_14 = StandardScaler()

X_train_14_scaled = scaler_14.fit_transform(X_train_14_imp)
X_val_14_scaled = scaler_14.transform(X_val_14_imp)
X_test_14_scaled = scaler_14.transform(X_test_14_imp)

print("14-feature dataset prepared.")
print("Training:", X_train_14_scaled.shape)
print("Validation:", X_val_14_scaled.shape)
print("Test:", X_test_14_scaled.shape)

In [ ]:
# ===== 14-FEATURE RANDOM FOREST =====

rf_14 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Train
rf_14.fit(X_train_14_scaled, y_train_14)

# Validation prediction
y_val_pred_14 = rf_14.predict(X_val_14_scaled)

# Metrics
mae_14 = mean_absolute_error(y_val_14, y_val_pred_14)
rmse_14 = np.sqrt(
    mean_squared_error(y_val_14, y_val_pred_14)
)
r2_14 = r2_score(y_val_14, y_val_pred_14)

print("===== 14-FEATURE RANDOM FOREST =====")
print(f"Validation MAE  : {mae_14:.2f} knots")
print(f"Validation RMSE : {rmse_14:.2f} knots")
print(f"Validation R²   : {r2_14:.4f}")

print("\n===== COMPARISON =====")
print("Original 11-feature RF")
print("MAE  : 13.07 knots")
print("RMSE : 18.51 knots")
print("R²   : 0.6236")

print("\nNew 14-feature RF")
print(f"MAE  : {mae_14:.2f} knots")
print(f"RMSE : {rmse_14:.2f} knots")
print(f"R²   : {r2_14:.4f}")

In [ ]:
# ===== 14-FEATURE IMPORTANCE =====

importance_14 = pd.DataFrame({
    "Feature": FEATURES_14,
    "Importance": rf_14.feature_importances_
})

importance_14 = importance_14.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("===== 14-FEATURE IMPORTANCE =====")
print(importance_14.to_string(index=False))

In [ ]:
# ===== FINAL TEST EVALUATION =====

# Predict on the untouched test set
y_test_pred_14 = rf_14.predict(X_test_14_scaled)

# Calculate test metrics
test_mae = mean_absolute_error(
    y_test_14,
    y_test_pred_14
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test_14,
        y_test_pred_14
    )
)

test_r2 = r2_score(
    y_test_14,
    y_test_pred_14
)

print("===== FINAL WIND INTENSITY TEST RESULTS =====")
print(f"Test MAE  : {test_mae:.2f} knots")
print(f"Test RMSE : {test_rmse:.2f} knots")
print(f"Test R²   : {test_r2:.4f}")

print("\n===== TRAIN / VALIDATION / TEST =====")
print("Validation MAE :", f"{mae_14:.2f}", "knots")
print("Validation RMSE:", f"{rmse_14:.2f}", "knots")
print("Validation R²  :", f"{r2_14:.4f}")

print("Test MAE       :", f"{test_mae:.2f}", "knots")
print("Test RMSE      :", f"{test_rmse:.2f}", "knots")
print("Test R²        :", f"{test_r2:.4f}")

In [ ]:
# ===== SAVE FINAL INTENSITY MODEL =====

import joblib

joblib.dump(rf_14, "final_wind_intensity_model.pkl")
joblib.dump(imputer_14, "wind_intensity_imputer.pkl")
joblib.dump(scaler_14, "wind_intensity_scaler.pkl")

print("Final wind-intensity model saved.")
print("Preprocessing objects saved.")